## Example 1: Basic Ray Task Runner Setup

In [ ]:
from prefect_ray import RayTaskRunner
from prefect import flow, task

# Example 1: Basic Ray Task Runner Setup
print("Example 1 - Basic Ray Task Runner Setup")

# Create RayTaskRunner with default configuration
ray_runner = RayTaskRunner()

print(f"RayTaskRunner created:")
print(f"  Type: {type(ray_runner).__name__}")
print(f"  Address: {ray_runner.address}")
print(f"  Init kwargs: {ray_runner.init_kwargs}")

# Define a simple flow using RayTaskRunner
@flow(task_runner=RayTaskRunner())
def basic_ray_flow():
    """Simple flow using Ray task runner"""
    return "Flow using RayTaskRunner"

print(f"\nFlow configuration:")
print(f"  Flow name: {basic_ray_flow.name}")
print(f"  Task runner: {type(basic_ray_flow.task_runner).__name__}")

from prefect.task_runners import TaskRunner
assert isinstance(ray_runner, TaskRunner)
# With no address, the runner starts a local Ray instance when a flow runs; Prefect adds its namespace
assert ray_runner.address is None
assert ray_runner.init_kwargs == {"namespace": "prefect"}
assert basic_ray_flow.name == "basic-ray-flow"
assert isinstance(basic_ray_flow.task_runner, RayTaskRunner)

print("\nExample 1 completed successfully")

## Example 2: Ray Task Runner with Custom Configuration

In [ ]:
from prefect_ray import RayTaskRunner
from prefect import flow, task

# Example 2: Ray Task Runner with Custom Configuration
print("Example 2 - Ray Task Runner with Custom Configuration")

# Configure RayTaskRunner with custom init_kwargs
custom_runner = RayTaskRunner(
    init_kwargs={
        "num_cpus": 4,
        "ignore_reinit_error": True,
        "include_dashboard": False
    }
)

print(f"Custom RayTaskRunner configuration:")
print(f"  Init kwargs: {custom_runner.init_kwargs}")
print(f"  Num CPUs: {custom_runner.init_kwargs.get('num_cpus')}")
print(f"  Ignore reinit error: {custom_runner.init_kwargs.get('ignore_reinit_error')}")

# Create flow with custom Ray configuration
@flow(task_runner=custom_runner)
def configured_ray_flow():
    """Flow with custom Ray configuration"""
    return "Configured Ray flow"

print(f"\nFlow with custom task runner:")
print(f"  Flow: {configured_ray_flow.name}")

assert custom_runner.init_kwargs == {
    "num_cpus": 4, "ignore_reinit_error": True, "include_dashboard": False, "namespace": "prefect",
}
assert configured_ray_flow.task_runner is custom_runner
# duplicate() gives an equal runner with its own copy of the configuration
copy = custom_runner.duplicate()
assert copy is not custom_runner and copy == custom_runner
assert copy.init_kwargs == custom_runner.init_kwargs

print("\nExample 2 completed successfully")

## Example 3: Parallel Task Processing

In [ ]:
from prefect_ray import RayTaskRunner
from prefect import flow, task

# Example 3: Parallel Task Processing with Ray
print("Example 3 - Parallel Task Processing with Ray")

@task
def process_item(item):
    """Process a single item"""
    return f"Processed: {item}"

@flow(task_runner=RayTaskRunner(init_kwargs={"num_cpus": 2, "include_dashboard": False, "ignore_reinit_error": True}))
def parallel_processing_flow():
    """Flow with parallel task execution using Ray"""
    items = ["data_1", "data_2", "data_3", "data_4", "data_5"]
    
    # Submit tasks for parallel execution
    futures = [process_item.submit(item) for item in items]
    results = [future.result() for future in futures]
    
    return results

print(f"Parallel processing flow configuration:")
print(f"  Flow: {parallel_processing_flow.name}")
print(f"  Task runner: {type(parallel_processing_flow.task_runner).__name__}")

# Running the flow starts a local two-CPU Ray instance and runs the five tasks on it
results = parallel_processing_flow()
print(f"  Results: {results}")
assert results == [f"Processed: data_{i}" for i in range(1, 6)], results

print("\nExample 3 completed successfully")

## Example 4: Remote Ray Cluster Connection

In [ ]:
from prefect_ray import RayTaskRunner
from prefect import flow, task

# Example 4: Remote Ray Cluster Connection
print("Example 4 - Remote Ray Cluster Connection")

# Configure RayTaskRunner to connect to remote cluster
remote_runner = RayTaskRunner(
    address="ray://remote-cluster:10001",
    init_kwargs={
        "runtime_env": {"pip": ["numpy"]},
        "ignore_reinit_error": True
    }
)

print(f"Remote Ray cluster configuration:")
print(f"  Address: {remote_runner.address}")
print(f"  Runtime env: {remote_runner.init_kwargs.get('runtime_env')}")

# Flow targeting remote cluster
@flow(task_runner=remote_runner)
def remote_cluster_flow():
    """Flow executing on remote Ray cluster"""
    return "Remote execution"

print(f"\nRemote cluster flow:")
print(f"  Flow: {remote_cluster_flow.name}")
print(f"  Target address: {remote_cluster_flow.task_runner.address}")

# Only the configuration is checked here; nothing connects to the address
assert remote_runner.address == "ray://remote-cluster:10001"
assert remote_runner.init_kwargs["runtime_env"] == {"pip": ["numpy"]}
assert remote_runner.init_kwargs["ignore_reinit_error"] is True
assert remote_cluster_flow.task_runner.address == remote_runner.address

print("\nExample 4 completed successfully")

## Example 5: Production Ray Configuration

In [ ]:
from prefect_ray import RayTaskRunner
from prefect import flow, task

# Example 5: Production Ray Configuration
print("Example 5 - Production Ray Configuration")

# Production-grade RayTaskRunner configuration
production_runner = RayTaskRunner(
    init_kwargs={
        "num_cpus": 8,
        "num_gpus": 0,
        "object_store_memory": 2 * 1024**3,  # 2GB
        "ignore_reinit_error": True,
        "include_dashboard": True,
        "dashboard_host": "0.0.0.0",
        "logging_level": "info"
    }
)

print(f"Production RayTaskRunner configuration:")
for key, value in production_runner.init_kwargs.items():
    print(f"  {key}: {value}")

@flow(task_runner=production_runner, name="production-ray-pipeline")
def production_pipeline():
    """Production pipeline with Ray task runner"""
    return {"status": "configured", "runner": "RayTaskRunner"}

print(f"\nProduction flow: {production_pipeline.name}")

assert production_runner.address is None
assert production_runner.init_kwargs["num_cpus"] == 8
assert production_runner.init_kwargs["object_store_memory"] == 2 * 1024**3
assert production_runner.init_kwargs["dashboard_host"] == "0.0.0.0"
assert production_runner.init_kwargs["namespace"] == "prefect"
assert production_pipeline.name == "production-ray-pipeline"
assert production_pipeline.task_runner is production_runner

print("\nExample 5 completed successfully")